# 3. Grading from the volumetric features

Reproduces **Section 3.5** (the classifier panel), **Section 3.6** (the protocol) and
**Section 4.3** (the results):

| output | paper |
|---|---|
| six classifiers, accuracy, F1 and AUC, n = 369 | `tab:clf` |
| the panel extended with XGBoost, histogram gradient boosting and regularized logistic regression | Section 4.3 |
| the label-shuffle control | Section 4, "shuffling the grade labels drops the random forest to an AUC of 0.43" |

The SHAP attributions (`fig:shap`) are drawn in notebook 7, with the other figures.

Runs from `data_cache/` alone: no volumes, no models, a couple of minutes on a laptop.

## The protocol

Case-level five-fold stratified cross-validation, one row per case, scored by the area under the
ROC curve. The scaler sits inside the pipeline, so it is fit on training folds only. The same
protocol is used by every table in the paper.

In [ ]:
import sys, warnings
sys.path.insert(0, "..")
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

from src import config
print(config.describe())

In [ ]:
from src import cohort, evaluation, tables

predicted = cohort.extract_features()
grades = cohort.load_grade_labels()
X, y, cases = cohort.build_matrix(predicted, grades)
print(f"n = {len(cases)}  high-grade {int((y == 0).sum())}  low-grade {int(y.sum())}")
print(f"folds: {evaluation.CV}")

## The grading folds are the segmentation folds

The features below come from cross-fitted masks: the masks of each grading fold were predicted by
a MedSAM2 fine-tuned on the other four folds (notebook 1). That only holds if the fold of every
case in `data_cache/folds.csv` is the test fold this cross-validation puts it in, so it is checked
case by case rather than assumed.

In [ ]:
folds = cohort.check_folds(cases, y)
table = folds.groupby(["fold", "grade"]).size().unstack()
table["total"] = table.sum(axis=1)
display(table)
print("every case sits in the same fold for segmentation and for grading")

## tab:clf

Expected, on the 369-case cohort:

| Classifier | Accuracy | F1 | AUC |
|---|---|---|---|
| SVM (RBF) | 0.886 | 0.746 | **0.927** |
| SVM (linear) | 0.894 | 0.768 | 0.915 |
| Random forest | 0.908 | 0.790 | 0.925 |
| Gradient boosting | 0.905 | 0.753 | 0.925 |
| Logistic reg. | 0.892 | 0.762 | 0.907 |
| k-NN (k = 5) | 0.889 | 0.665 | 0.882 |

with XGBoost at 0.926 on the same features. The AUCs sit between 0.882 and 0.927, which is the
point: the result is a property of the features, not of one lucky model.

In [ ]:
clf = tables.table_clf(X, y)
display(clf)

published = {
    "SVM (RBF)": ("0.886", "0.746", "0.927"),
    "SVM (linear)": ("0.894", "0.768", "0.915"),
    "Random forest": ("0.908", "0.790", "0.925"),
    "Gradient boosting": ("0.905", "0.753", "0.925"),
    "Logistic reg.": ("0.892", "0.762", "0.907"),
    "$k$-NN ($k = 5$)": ("0.889", "0.665", "0.882"),
    "XGBoost": (None, None, "0.926"),
}
expected = {row: {col: v for col, v in zip(["Accuracy", "F1", "AUC"], values) if v}
            for row, values in published.items()}
check = tables.check_printed(clf, expected)
display(check)
print("all cells match the paper:", bool(check["match"].all()))

## The extended panel

Adding XGBoost, histogram gradient boosting and $L_2$- and elastic-net-regularized logistic
regression does not raise the AUC: the three added in Section 4.3 reach 0.920, 0.920 and 0.918,
all below the RBF SVM's 0.927. Widening the panel was a check against having picked a lucky family
of models, and it comes back negative.

In [ ]:
extended = tables.table_clf_extended(X, y)
display(extended.sort_values("AUC", ascending=False))
print(f"best AUC in the extended panel: {extended['AUC'].max():.3f}")

check = tables.check_printed(extended, {
    "XGBoost": {"AUC": "0.926"},
    "Hist. grad. boosting": {"AUC": "0.920"},
    "Logistic reg. ($L_2$)": {"AUC": "0.920"},
    "Logistic reg. (elastic net)": {"AUC": "0.918"},
})
display(check)
print("all cells match:", bool(check["match"].all()))

## Integrity control

With the grades permuted, the signal has to disappear. The random forest drops to **0.43**,
below chance, which is what a small cohort does when there is nothing left to learn.

In [ ]:
shuffled = evaluation.shuffled_label_control(X, y)
display(shuffled[["Accuracy", "F1", "AUC"]])
from src.stats import round_half_up
shuffled_rf = shuffled.loc["Random forest", "AUC"]
print(f"random forest on shuffled grades: {shuffled_rf:.3f}, "
      f"{round_half_up(shuffled_rf, 2):.2f} at the paper's precision "
      f"(paper 0.43, match {f'{round_half_up(shuffled_rf, 2):.2f}' == '0.43'})")